# BERT Fine-Tuning — IMDB Sentiment

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

BERT is a stack of encoder blocks pre-trained with masked language modeling and next-sentence prediction. To use it for classification we add a small head on top of the `[CLS]` representation and fine-tune the whole thing on the target task — here, binary sentiment on IMDB.


## Mathematical Formulation

For an input sequence we feed `[CLS] tokens [SEP]`. The final hidden state at position 0 is `h_{[CLS]}` and the logits are

$$\hat{y} = W\,h_{[CLS]} + b, \quad \mathcal{L} = \text{CrossEntropy}(\hat{y}, y).$$

We back-propagate through every layer; this is *fine-tuning*, not feature extraction.


## Implementation


In [ ]:
# Install once: pip install transformers datasets accelerate
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup


In [ ]:
model_name = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


In [ ]:
ds = load_dataset('imdb')

def encode(batch):
    return tok(batch['text'], truncation=True, padding='max_length', max_length=128)

train = ds['train'].shuffle(seed=0).select(range(2000)).map(encode, batched=True)
val   = ds['test'].shuffle(seed=0).select(range(500)).map(encode, batched=True)
train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_dl = DataLoader(train, batch_size=16, shuffle=True)
val_dl   = DataLoader(val, batch_size=32)


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=0, num_training_steps=len(train_dl) * 2)

for epoch in range(2):
    model.train()
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['label'])
        out.loss.backward()
        opt.step(); sched.step(); opt.zero_grad()
    print(f'epoch {epoch} train loss {out.loss.item():.3f}')


## Experiment


In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for batch in val_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
        correct += (logits.argmax(-1) == batch['label']).sum().item()
        total += batch['label'].size(0)
print(f'val accuracy: {correct/total:.3f}')


## Discussion

- Use a very small learning rate (2e-5) — BERT is already well-trained and you don't want to overwrite the pretrained features.
- Warmup + linear decay is the standard schedule for fine-tuning.
- For a real run, bump training to the full IMDB split and use a longer max length (256–512).


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
